
<div style="background-color: lightgreen; color: black; padding: 20px; box-sizing: border-box; width: 99%;">

👋 <b>Olá! Eu sou o Dot, seu revisor de IA.</b>

🎉 <b>Parabéns! Você foi aprovado no seu projeto.</b>

Excelente trabalho ao concluir esta tarefa, e obrigado por corrigir as contradições na sua explicação final: agora ela está consistente e corresponde aos seus resultados. Seu Random Forest com pesos de classe (class-weighted) alcançou um F1 de 0,632 e uma AUC-ROC de 0,862 no conjunto de teste, confortavelmente acima do valor mínimo exigido de 0,59. Para seu crescimento futuro, mantenha as saídas impressas e as conclusões escritas totalmente alinhadas (por exemplo, mostre claramente a linha do melhor threshold e mencione o empate com o ajuste de threshold), e use nomes de variáveis específicos para cada modelo, para que o código seja seguro de estender. Continue com o ótimo trabalho!

Se você tiver alguma dificuldade ou erro, fique à vontade para entrar em contato com nossa equipe de suporte clicando no ícone de chat no canto superior direito ou enviando um e-mail para [helpme@tripleten.com](mailto:helpme@tripleten.com).

<hr style="margin-top: 20px; margin-bottom: 20px;">

<details>
<summary><b style="cursor: pointer;">💡 Clique aqui para ver como ler meus comentários (Legenda de cores)</b></summary>
<div style="margin-top: 15px;">
Veja meus comentários abaixo. <b>Peço gentilmente que não os mova, modifique nem exclua</b>.

Você encontrará meus comentários destacados em caixas verdes, amarelas ou vermelhas como estas:

<div class="alert alert-block alert-success">
Sucesso. Tudo foi feito corretamente.
</div>

<div class="alert alert-block alert-warning">
Observações. Algumas recomendações.
</div>

<div class="alert alert-block alert-danger">
A corrigir. O bloco requer algumas correções.
</div>
</div>
</details>

</div>


In [1]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_score
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [2]:
df = pd.read_csv('/datasets/Churn.csv')
print(df.head())
print(df.info())
print(df.duplicated().sum())
print(df.isna().sum())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0     2.0       0.00              1          1               1   
1     1.0   83807.86              1          0               1   
2     8.0  159660.80              3          1               0   
3     1.0       0.00              2          0               0   
4     2.0  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

In [3]:
print(df['Exited'].value_counts(normalize=True))
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
print(df[df['Tenure'].isna()])
print(df['Geography'].unique())
print(df['Gender'].unique())
data = pd.get_dummies(df, drop_first=True)
print(data.head())

0    0.7963
1    0.2037
Name: Exited, dtype: float64
      CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
30            591     Spain  Female   39     NaN       0.00              3   
48            550   Germany    Male   38     NaN  103391.38              1   
51            585   Germany    Male   36     NaN  146050.97              2   
53            655   Germany    Male   41     NaN  125561.97              1   
60            742   Germany    Male   35     NaN  136857.00              1   
...           ...       ...     ...  ...     ...        ...            ...   
9944          744   Germany    Male   41     NaN  190409.34              2   
9956          520    France  Female   46     NaN   85216.61              1   
9964          479    France    Male   34     NaN  117593.48              2   
9985          659    France    Male   36     NaN  123841.49              2   
9999          792    France  Female   28     NaN  130142.79              1   

      HasC

Importamos os dados nessa parte acima. O Tenure com 909 valores ausentes será substituído pela mediana da coluna, para que esses dados não prejudiquem o treinamento. A mediana foi escolhida por ser uma medida robusta contra valores extremos. Também removi as colunas RowNumber, CustomerId e Surname, pois são identificadores sem valor preditivo. Também vamos usar o get_dummies por causa das colunas Geography e Gender. Podemos ver que temos uma saída de cerca de 20% dos clientes e permanecem 80%, então há um desbalanceamento entre as classes. Um modelo "burro" que chuta que ninguém sai acertaria 80% das vezes (acurácia alta), mas seria péssimo para o banco, pois não identificaria nenhum cliente que realmente vai sair, e o Recall e o F1 seriam exatamente 0. O parâmetro drop_first é utilizado em True para eliminar uma coluna redundante. Remover uma coluna dummy evita a "armadilha da variável dummy" (multicolinearidade perfeita), já que a última categoria pode ser inferida a partir das outras. Isso é especialmente relevante para modelos lineares, como a Regressão Logística. A mediana é calculada apenas a partir dos dados de treino, para evitar data leakage (vazamento de dados).

<div class="alert alert-block alert-success">

Sua explicação das etapas de preparação está clara, e cada decisão tem uma justificativa por trás.
- Você escolheu a mediana para `Tenure` porque ela é robusta a valores extremos.
- Você removeu as colunas de identificadores porque elas não têm valor preditivo.
- Você explicou o `drop_first=True` e a armadilha da variável dummy (dummy variable trap).
- Você mostrou por que um modelo que prevê que "ninguém sai" alcançaria cerca de 80% de acurácia, mas teria recall e F1 iguais a zero.

</div>

In [4]:
features = data.drop(['Exited'], axis=1)
target = data['Exited']

features_train_valid, features_test, target_train_valid, target_test = train_test_split(
    features, target, test_size=0.20, random_state=12345, stratify=target
)
# SEGUNDO CORTE: Do restante (80%), separar 25% para Validação
# Isso resulta em 20% do total para validação (25% de 80% = 20%)
features_train, features_valid, target_train, target_valid = train_test_split(
    features_train_valid, target_train_valid, test_size=0.25, random_state=12345, stratify=target_train_valid
)

features_train = features_train.copy()
features_valid = features_valid.copy()
features_test = features_test.copy()
mediana_tenure = features_train['Tenure'].median()

features_train['Tenure'] = features_train['Tenure'].fillna(mediana_tenure)
features_valid['Tenure'] = features_valid['Tenure'].fillna(mediana_tenure)
features_test['Tenure'] = features_test['Tenure'].fillna(mediana_tenure)

numeric = ['Balance', 'EstimatedSalary', 'CreditScore']
scaler = StandardScaler()
scaler.fit(features_train[numeric])
features_train[numeric] = scaler.transform(features_train[numeric])
features_valid[numeric] = scaler.transform(features_valid[numeric])
features_test[numeric] = scaler.transform(features_test[numeric])

<div class="alert alert-block alert-success">

Sua divisão dos dados e o pré-processamento estão sólidos.
- O `train_test_split` estratificado em duas etapas gera uma divisão 60/20/20 e mantém a taxa de churn próxima de 20% em todos os conjuntos.
- Você calculou a mediana de `Tenure` e ajustou o `StandardScaler` apenas no conjunto de treino, e depois os aplicou aos conjuntos de validação e teste. Isso evita data leakage (vazamento de dados).
- Usar `.copy()` antes das atribuições é um bom hábito.

</div>

Dados separados para treino, teste e validação. Sendo 20% para teste, 20% para validação e 60% para treino. O conjunto de validação é usado para ajustar e comparar modelos e técnicas, já o conjunto de teste permanece intocado até a avaliação final. O StandardScaler foi realizado nas seguintes colunas: 'Balance', 'EstimatedSalary', 'CreditScore'. Esse procedimento é realizado para padronizar a escala das variáveis núméricas. O StandardScaler foi ajustado (fit) apenas no conjunto de treino para evitar data leakage (vazamento de dados). Se o scaler fosse ajustado com todos os dados (treino + validação + teste), ele calcularia a média e o desvio padrão usando informações dos dados de teste, que o modelo não deveria conhecer. Isso faria com que a avaliação do modelo ficasse artificialmente otimista, pois o modelo teria 'visto' indiretamente os dados de teste durante o pré-processamento. Ao ajustar o scaler apenas no treino e aplicar (transform) nos demais conjuntos, garantimos que a avaliação seja honesta e reflita o desempenho real do modelo em dados nunca vistos.

In [5]:

print(' ----------- Decision Tree Classifier')
best_f1_dt, best_depth_dt = 0, 0
for depth in range(1, 21):
    model_dt = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model_dt.fit(features_train, target_train)
    f1 = f1_score(target_valid, model_dt.predict(features_valid))
    if f1 > best_f1_dt:
        best_f1_dt, best_depth_dt = f1, depth

model_dt = DecisionTreeClassifier(random_state=12345, max_depth=best_depth_dt)
model_dt.fit(features_train, target_train)
auc_dt = roc_auc_score(target_valid, model_dt.predict_proba(features_valid)[:, 1])
print('Decision Tree | depth:', best_depth_dt, '| F1:', best_f1_dt, '| AUC-ROC:', auc_dt)

print(' ----------- Logistic Regression com liblinear')
model_lr = LogisticRegression(random_state=12345, solver='liblinear')
model_lr.fit(features_train, target_train)
predictions_valid = model_lr.predict(features_valid)
print('F1 (Logistic Regression, liblinear):', f1_score(target_valid, predictions_valid, zero_division=0))

probabilities_valid = model_lr.predict_proba(features_valid)[:, 1]
auc_roc = roc_auc_score(target_valid, probabilities_valid)
print("AUC-ROC (Logistic Regression, liblinear):", auc_roc)

print(' ----------- Logistic Regression sem liblinear')
model_lr2 = LogisticRegression(random_state=12345, class_weight='balanced', max_iter=1000)
model_lr2.fit(features_train, target_train)
predictions_valid = model_lr2.predict(features_valid)
print('F1 (Logistic Regression, class_weight=balanced):', f1_score(target_valid, predictions_valid, zero_division=0))

probabilities_valid = model_lr2.predict_proba(features_valid)[:, 1]
auc_roc = roc_auc_score(target_valid, probabilities_valid)
print("AUC-ROC (Logistic Regression, class_weight=balanced):", auc_roc)


print(' ----------- Random Forest Classifier')
best_est = 0
best_f1 = 0
best_depth = 0

for n_estimators in range(1, 101, 10):
    for depth in range(1, 41):
        modelRFC = RandomForestClassifier(random_state=12345, n_estimators=n_estimators, max_depth=depth)
        modelRFC.fit(features_train, target_train)
        predictions_valid = modelRFC.predict(features_valid)
        f1 = f1_score(target_valid, predictions_valid, zero_division=0) 
        if f1 > best_f1:
            best_f1 = f1
            best_est = n_estimators
            best_depth = depth
        

print("Random Forest | melhor n_estimators:", best_est)
print("Random Forest | melhor max_depth:", best_depth)

modelRFC = RandomForestClassifier(random_state=12345, n_estimators=best_est, max_depth=best_depth)
modelRFC.fit(features_train, target_train)
predicted_valid = modelRFC.predict(features_valid)
print("Recall (Random Forest):", recall_score(target_valid, predicted_valid))
print("Precisão (Random Forest):", precision_score(target_valid, predicted_valid))
print("Matriz de confusão (Random Forest):", confusion_matrix(target_valid, predicted_valid))
probabilities_valid = modelRFC.predict_proba(features_valid)[:, 1]
auc_roc = roc_auc_score(target_valid, probabilities_valid)
print("AUC-ROC (Random Forest):", auc_roc)
print('F1 (Random Forest):', f1_score(target_valid, predicted_valid))

 ----------- Decision Tree Classifier
Decision Tree | depth: 8 | F1: 0.5796269727403156 | AUC-ROC: 0.8073967650238837
 ----------- Logistic Regression com liblinear
F1 (Logistic Regression, liblinear): 0.3189493433395873
AUC-ROC (Logistic Regression, liblinear): 0.7873559229491434
 ----------- Logistic Regression sem liblinear
F1 (Logistic Regression, class_weight=balanced): 0.5113043478260869
AUC-ROC (Logistic Regression, class_weight=balanced): 0.7918411477733514
 ----------- Random Forest Classifier
Random Forest | melhor n_estimators: 81
Random Forest | melhor max_depth: 24
Recall (Random Forest): 0.4742014742014742
Precisão (Random Forest): 0.7568627450980392
Matriz de confusão (Random Forest): [[1531   62]
 [ 214  193]]
AUC-ROC (Random Forest): 0.852056987650208
F1 (Random Forest): 0.5830815709969789


<div class="alert alert-block alert-warning">

Os baselines estão muito bem feitos. Você ajustou os hiperparâmetros apenas no conjunto de validação e reportou F1 e AUC-ROC para todos os modelos, além de recall, precision e a matriz de confusão para o Random Forest. Duas pequenas sugestões opcionais:
- A seção impressa como "Logistic Regression sem liblinear" na verdade usa `class_weight='balanced'`. Um rótulo como "Logistic Regression com class_weight='balanced'" ficaria alinhado com o código e com a sua explicação.
- `predictions_valid`, `probabilities_valid` e `auc_roc` são sobrescritos a cada modelo. Nomes específicos para cada modelo, como `probabilities_valid_rf`, tornam mais seguro escrever as células seguintes.

</div>


Resultados obtidos na validação (modelos sem tratamento de desequilíbrio, exceto onde indicado):

Árvore de Decisão (max_depth=8): F1 0.580 / AUC-ROC 0.807
Regressão Logística sem class_weight: F1 0.319 / AUC-ROC 0.787
Regressão Logística com class_weight='balanced': F1 0.511 / AUC-ROC 0.792
Random Forest (n_estimators=81, max_depth=24): F1 0.583 / AUC-ROC 0.852
A Regressão Logística sem ponderação tem um F1 muito baixo (0.319) apesar de um AUC-ROC razoável (0.787): o modelo ordena os clientes de forma aceitável, mas o limiar padrão de 0.5 quase nunca marca a classe minoritária. Com class_weight='balanced' o F1 sobe para 0.511, mas continua abaixo da Árvore de Decisão e da Random Forest.

A melhor combinação de hiperparâmetros da Random Forest, escolhida na validação, foi n_estimators=81 e max_depth=24, com F1 de 0.583, ligeiramente abaixo da marca de 0.59. A precisão é alta (0.76), mas o recall é de apenas 0.47: 214 churners não detectados contra 193 detectados corretamente. O modelo é conservador demais, um efeito típico do desequilíbrio de classes, e o recall é o gargalo; o F1 equilibra os dois. O AUC-ROC (0.852) é bem maior que o F1 porque mede o ranqueamento em todos os limiares, enquanto o F1 depende do limiar fixo de 0.5.

Como a Random Forest teve o maior F1 e o maior AUC-ROC, seguimos com ela e passamos para as técnicas de correção do desequilíbrio.

<div class="alert alert-block alert-success">

Sua análise dos resultados de baseline está clara, e todos os números correspondem às saídas impressas.
- Você explicou corretamente por que a Regressão Logística simples tem um F1 muito baixo (0.319), apesar de uma AUC-ROC razoável (0.787): o threshold padrão de 0.5 quase nunca sinaliza a classe minoritária.
- Você identificou o recall (0.47) como o gargalo do Random Forest, com respaldo na matriz de confusão (214 churners não detectados contra 193 identificados).
- Você explicou por que a AUC-ROC (0.852) é maior que o F1 (0.583): ela mede o ranqueamento considerando todos os thresholds, enquanto o F1 depende de um único threshold fixo.
- A escolha do Random Forest como modelo base está bem justificada.

</div>

Então vamos testar algumas abordagens para melhorar o nosso modelo. Vamos usar o class_weight='balanced' no construtor do seu modelo, isso força o modelo a dar mais valor para a classe 1.

In [6]:


best_f1_w, best_est_w, best_depth_w = 0, 0, 0
for n_est in range(10, 101, 10):
    for depth in range(1, 21):
        m = RandomForestClassifier(random_state=12345, class_weight='balanced',
                                   n_estimators=n_est, max_depth=depth)
        m.fit(features_train, target_train)
        f1 = f1_score(target_valid, m.predict(features_valid))
        if f1 > best_f1_w:
            best_f1_w, best_est_w, best_depth_w = f1, n_est, depth
print("Melhor est:", best_est_w," Melhor depth:", best_depth_w," Melhor F1:", best_f1_w)


model = RandomForestClassifier(random_state=12345, class_weight='balanced' , n_estimators=best_est_w, max_depth=best_depth_w)
model.fit(features_train, target_train)
predicted_valid = model.predict(features_valid)
print("Sensibilidade do sistema: ", recall_score(target_valid, predicted_valid))  
print("Precisão: ", precision_score(target_valid, predicted_valid))
print("Matriz de confusão: ", confusion_matrix(target_valid, predicted_valid))
probabilities_valid = model.predict_proba(features_valid)[:, 1]
auc_roc = roc_auc_score(target_valid, probabilities_valid)
print("AUC-ROC:", auc_roc)
print('F1:', f1_score(target_valid, predicted_valid))

Melhor est: 60  Melhor depth: 10  Melhor F1: 0.6431181485992692
Sensibilidade do sistema:  0.6486486486486487
Precisão:  0.6376811594202898
Matriz de confusão:  [[1443  150]
 [ 143  264]]
AUC-ROC: 0.8628412696209307
F1: 0.6431181485992692


<div class="alert alert-block alert-success">

Bom trabalho com o balanceamento de classes (class weighting).
- Você executou uma busca dedicada para a floresta aleatória ponderada, em vez de reutilizar os parâmetros do modelo sem ponderação.
- Você selecionou o modelo apenas com base no conjunto de validação.
- O modelo retreinado (`n_estimators=60`, `max_depth=10`) corresponde ao resultado da busca. O F1 sobe de 0,583 para 0,643, o recall de 0,47 para 0,649, e a AUC-ROC é 0,863.
- Opcional: um nome específico, como `model_rf_weighted`, é mais seguro do que o genérico `model`, que poderia ser sobrescrito mais tarde.

</div>

Vamos fazer um upsampling que é uma função que duplica os dados de quem saiu no conjunto de treinamento. 

In [7]:

best_f1_up = 0
best_est_up = 0
best_depth_up = 0
best_repeat_up = 0

def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]
    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)
    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345
    )
    return features_upsampled, target_upsampled

for repeat in [2, 3, 4, 5, 10]:
    print(f'Repeat = {repeat} -----------------------')
    features_upsampled, target_upsampled = upsample(
        features_train, target_train, repeat
    )
    modelUP = RandomForestClassifier(random_state=12345, n_estimators=best_est_w, max_depth=best_depth_w)
    modelUP.fit(features_upsampled, target_upsampled)
    predicted_valid = modelUP.predict(features_valid)
    probabilities_valid = modelUP.predict_proba(features_valid)[:, 1]
    auc_roc = roc_auc_score(target_valid, probabilities_valid)

    print("AUC-ROC:", auc_roc)
    print('F1:', f1_score(target_valid, predicted_valid))
    print('Precisão:', precision_score(target_valid, predicted_valid))
    print('Recall:', recall_score(target_valid, predicted_valid))




Repeat = 2 -----------------------
AUC-ROC: 0.8678169695118848
F1: 0.6134800550206327
Precisão: 0.696875
Recall: 0.547911547911548
Repeat = 3 -----------------------
AUC-ROC: 0.8666601886940869
F1: 0.6365795724465559
Precisão: 0.6160919540229886
Recall: 0.6584766584766585
Repeat = 4 -----------------------
AUC-ROC: 0.8629168459676935
F1: 0.6323851203501095
Precisão: 0.5700197238658777
Recall: 0.7100737100737101
Repeat = 5 -----------------------
AUC-ROC: 0.8665984937171378
F1: 0.6106106106106106
Precisão: 0.5152027027027027
Recall: 0.7493857493857494
Repeat = 10 -----------------------
AUC-ROC: 0.8597919953852158
F1: 0.5522388059701493
Precisão: 0.4167709637046308
Recall: 0.8181818181818182


<div class="alert alert-block alert-success">

Seu upsampling está bem implementado.
- `upsample` repete apenas a classe minoritária, embaralha com uma seed fixa e é aplicado somente ao conjunto de treino, então nada vaza para a validação.
- Você testou vários valores de `repeat` com os mesmos hiperparâmetros da floresta usados no modelo ponderado, então a comparação é justa.
- Você imprime F1, precisão, recall e AUC-ROC para cada valor, o que deixa o trade-off visível. A melhor execução é `repeat=3` (F1 0.637).
- Esta é uma segunda técnica válida para lidar com o desbalanceamento de classes.
- Opcional: as variáveis `best_*_up` no início nunca são usadas, então você pode excluí-las.

</div>


Vamos comparar os nossos resultados até o momento:

class_weight='balanced' AUC-ROC: 0.8628412696209307 F1: 0.6431181485992692

Repeat = 4 -----------------------
AUC-ROC: 0.8629168459676935
F1: 0.6323851203501095
Precisão: 0.5700197238658777
Recall: 0.7100737100737101

Repeat = 10 -----------------------
AUC-ROC: 0.8597919953852158
F1: 0.5522388059701493
Precisão: 0.4167709637046308
Recall: 0.8181818181818182

Podemos ver que ao aumentar o repeat para 10 os dados saíram abaixo do repeat para 4. Contudo, o F1 mais alto veio do class_weight='balanced'. O que está acontecendo é que como os churners são duplicados muitas vezes o modelo acaba ficando "viciado" em prever churns. Ao ponto de estar quase pegando todos os  churners. Só que está prevendo churns para clientes fieis dando falsos positivos, no que diminui a precisão. 

<div class="alert alert-block alert-warning">

Seu raciocínio está correto: duplicar os churners muitas vezes faz o modelo prever churn em excesso, então o recall sobe enquanto a precisão e o F1 caem. Porém, a comparação deixa de fora a sua melhor execução de upsampling. Sua saída mostra `repeat=3` como o melhor (F1 0.637, precisão 0.616, recall 0.658, AUC-ROC 0.867), mas o texto compara apenas `repeat=4` e `repeat=10`. Melhorias opcionais:
- Compare com `repeat=3`, por exemplo: "`repeat=10` tem um F1 menor (0.552) do que `repeat=3` (0.637) porque a precisão cai para 0.417".
- "os dados saíram abaixo do repeat para 4" não está claro, então reescreva com números, como no exemplo acima.
- "quase pegando todos os churners" exagera um recall de 0.82. Escreva "o recall sobe para 0.82" no lugar.
- Adicione uma frase sobre AUC-ROC versus F1: a AUC-ROC quase não se move (0.860 para 0.868) enquanto o F1 oscila de 0.55 para 0.64, porque a AUC-ROC mede o ranqueamento em todos os limiares (thresholds), enquanto o F1 depende de um único limiar fixo.
- Escreva "fiéis" com o acento.

</div>

In [8]:
def downsample(features, target, fraction):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]
    features_downsampled = pd.concat(
        [features_zeros.sample(
            frac=fraction, random_state=12345)] + [features_ones]
    )
    target_downsampled = pd.concat(
        [target_zeros.sample(
            frac=fraction, random_state=12345)] + [target_ones]
    )
    features_downsampled, target_downsampled = shuffle(
        features_downsampled, target_downsampled, random_state=12345
    )
    return features_downsampled, target_downsampled

for fraction in [0.2, 0.3, 0.4, 0.5, 0.6]:
    features_down, target_down = downsample(features_train, target_train, fraction)
    model_down = RandomForestClassifier(random_state=12345, n_estimators=best_est_w, max_depth=best_depth_w)
    model_down.fit(features_down, target_down)
    predicted_valid = model_down.predict(features_valid)
    probabilities_valid = model_down.predict_proba(features_valid)[:, 1]
    auc_roc = roc_auc_score(target_valid, probabilities_valid)
    print("Fraction: ",fraction, " F1:",f1_score(target_valid, model_down.predict(features_valid)))
    print("AUC-ROC:", auc_roc)
    print('Precisão:', precision_score(target_valid, predicted_valid))
    print('Recall:', recall_score(target_valid, predicted_valid))


Fraction:  0.2  F1: 0.5659722222222222
AUC-ROC: 0.8580244342956207
Precisão: 0.4375838926174497
Recall: 0.800982800982801
Fraction:  0.3  F1: 0.6330178759200842
AUC-ROC: 0.8714909053892105
Precisão: 0.5533088235294118
Recall: 0.7395577395577395
Fraction:  0.4  F1: 0.6261904761904761
AUC-ROC: 0.8689845469506486
Precisão: 0.6073903002309469
Recall: 0.6461916461916462
Fraction:  0.5  F1: 0.6282722513089005
AUC-ROC: 0.8722219908660587
Precisão: 0.6722689075630253
Recall: 0.5896805896805897
Fraction:  0.6  F1: 0.5985611510791368
AUC-ROC: 0.8683367496926819
Precisão: 0.7222222222222222
Recall: 0.5110565110565111


<div class="alert alert-block alert-success">

Seu experimento de downsampling está bem construído.
- `downsample` reduz apenas a classe majoritária, mantém todos os clientes que cancelaram (churners), usa uma seed fixa e é aplicado somente ao conjunto de treinamento.
- Você testou cinco valores de `fraction` com os mesmos hiperparâmetros da floresta com pesos, então a comparação é justa.
- O melhor resultado é `fraction=0.3` (F1 0,633, AUC-ROC 0,871). Ele fica abaixo do balanceamento por pesos de classe, mas bem acima de 0,59, e é uma segunda técnica válida para lidar com o desbalanceamento de classes.
- Opcional: `predicted_valid` já foi calculado, então você pode reutilizá-lo no print do F1 em vez de chamar `predict` novamente.

</div>

Vamos comparar os nossos resultados até o momento:

class_weight='balanced' F1: 0.643

Fraction:  0.3  F1: 0.633

Não obteve um resultado superior ao class_weight='balanced' . O fraction 0.2 é pior por que remove fiéis demais, deixando a classe majoritária muito pequena. 


In [9]:

model_thr = RandomForestClassifier(random_state=12345, class_weight='balanced',
                                   n_estimators=best_est_w, max_depth=best_depth_w)
model_thr.fit(features_train, target_train)
probabilities_one_valid = model_thr.predict_proba(features_valid)[:, 1]

best_threshold, best_f1_thr = 0, 0
for threshold in np.arange(0.2, 0.7, 0.02):
    predicted_valid = probabilities_one_valid > threshold
    precision = precision_score(target_valid, predicted_valid)
    recall = recall_score(target_valid, predicted_valid)
    f1 = f1_score(target_valid, predicted_valid)
    print('Limiar = {:.2f} | Precisão = {:.3f}, Sensibilidade = {:.3f}, F1 = {:.3f}'.format(
        threshold, precision, recall, f1))
    if f1 > best_f1_thr:
        best_f1_thr, best_threshold = f1, threshold
print('Melhor limiar:', best_threshold, '| F1:', best_f1_thr)


Limiar = 0.20 | Precisão = 0.313, Sensibilidade = 0.929, F1 = 0.469
Limiar = 0.22 | Precisão = 0.331, Sensibilidade = 0.914, F1 = 0.486
Limiar = 0.24 | Precisão = 0.349, Sensibilidade = 0.892, F1 = 0.502
Limiar = 0.26 | Precisão = 0.366, Sensibilidade = 0.867, F1 = 0.515
Limiar = 0.28 | Precisão = 0.385, Sensibilidade = 0.855, F1 = 0.531
Limiar = 0.30 | Precisão = 0.404, Sensibilidade = 0.838, F1 = 0.545
Limiar = 0.32 | Precisão = 0.425, Sensibilidade = 0.823, F1 = 0.561
Limiar = 0.34 | Precisão = 0.442, Sensibilidade = 0.799, F1 = 0.569
Limiar = 0.36 | Precisão = 0.463, Sensibilidade = 0.784, F1 = 0.582
Limiar = 0.38 | Precisão = 0.484, Sensibilidade = 0.776, F1 = 0.596
Limiar = 0.40 | Precisão = 0.507, Sensibilidade = 0.757, F1 = 0.607
Limiar = 0.42 | Precisão = 0.522, Sensibilidade = 0.730, F1 = 0.609
Limiar = 0.44 | Precisão = 0.539, Sensibilidade = 0.710, F1 = 0.613
Limiar = 0.46 | Precisão = 0.573, Sensibilidade = 0.683, F1 = 0.623
Limiar = 0.48 | Precisão = 0.602, Sensibilidade 

<div class="alert alert-block alert-warning">

Bom ajuste de limiar (threshold) no modelo certo: a floresta com pesos por classe (class-weighted forest), treinada no conjunto de treino e avaliada apenas na validação. A saída impressa está cortada antes da linha 0.50 e antes da linha final "Melhor limiar", então o leitor não consegue confirmar o valor 0.50 / 0.643 que você cita no resumo. Melhorias opcionais:
- Imprima apenas as linhas próximas ao ponto ótimo (cerca de 0.44 a 0.56).
- Imprima o melhor resultado uma única vez após o loop, por exemplo `print('Melhor limiar: {:.2f} | F1: {:.3f}'.format(best_threshold, best_f1_thr))`. Formatar o número também evita artefatos de ponto flutuante gerados por `np.arange`.

</div>

Melhor F1 de validação por técnica: ponderação de classes 0.643, upsampling (repeat=3) 0.637, downsampling (fraction=0.3) 0.633, ajuste de limiar 0.50 (Limiar = 0.50 | Precisão = 0.638, Sensibilidade = 0.649, F1 = 0.643). A melhor abordagem é a ponderação de classes, por isso é a que uso no teste final.

In [10]:

features_up, target_up = upsample(features_train, target_train, 10) 

final_model = RandomForestClassifier(random_state=12345, class_weight='balanced',
                                     n_estimators=best_est_w, max_depth=best_depth_w)
final_model.fit(features_train, target_train)

predicted_test = final_model.predict(features_test)
probabilities_test = final_model.predict_proba(features_test)[:, 1]

print('F1 (test):', f1_score(target_test, predicted_test))
print('AUC-ROC (test):', roc_auc_score(target_test, probabilities_test))
print('Recall (test):', recall_score(target_test, predicted_test))
print('Precision (test):', precision_score(target_test, predicted_test))


F1 (test): 0.6317073170731707
AUC-ROC (test): 0.8617477261545059
Recall (test): 0.6363636363636364
Precision (test): 0.6271186440677966


<div class="alert alert-block alert-warning">

Seu teste final está configurado corretamente. `final_model` é a floresta aleatória com pesos de classe (`n_estimators=60`, `max_depth=10`), treinada apenas no conjunto de treino e avaliada uma única vez no conjunto de teste. Seus resultados são F1 0,632, AUC-ROC 0,862, recall 0,636 e precisão 0,627. O F1 está claramente acima de 0,59, e você reporta tanto o F1 quanto o AUC-ROC. Uma limpeza opcional: `features_up, target_up = upsample(features_train, target_train, 10)` cria dados que nunca são usados e sugere que um modelo com upsampling está envolvido quando não está, então você pode removê-la.

</div>

No conjunto de teste obtivemos um F1 de 0.632 (acima da meta de 0.59), um recall de 0.636 e uma precisão de 0.627. O AUC-ROC no teste obteve o valor de 0.862. A AUC-ROC não depende de um limiar, enquanto o F1 é calculado no limiar padrão de 0.5. A AUC-ROC é muito maior que o F1 porque mede o quão bem os clientes são ranqueados em todos os limiares, enquanto o F1 depende de um único limiar fixo e foca na classe de churn.

Comparando as técnicas na validação: ponderação de classes 0.643, upsampling (repeat=3) 0.637, downsampling (fraction=0.3) 0.633 e ajuste de limiar (melhor em 0.50) 0.643. As diferenças são pequenas, e a ponderação de classes venceu sem precisar reamostrar os dados. O ajuste de limiar foi avaliado apenas na validação e não melhorou o limiar padrão de 0.50, então o conjunto de teste foi usado somente para o modelo com ponderação de classes.

O modelo final é uma Random Forest com class_weight='balanced', n_estimators=60 e max_depth=10, que obteve F1 0.643 na validação e 0.632 no teste, com AUC-ROC de 0.862 no teste, superando as demais técnicas na validação.

<div class="alert alert-block alert-warning">

Sua explicação final está muito mais clara do que antes, e as contradições anteriores foram corrigidas.
- Ela usa um único valor de F1 (0,632) com três casas decimais do início ao fim.
- Ela relata um único resultado de limiar (o melhor em 0,50).
- Ela afirma que o ajuste de limiar foi avaliado apenas na validação e que o conjunto de teste foi usado somente para o modelo com ponderação de classes.
- A explicação de por que a AUC-ROC é maior que o F1 está correta.

Um pequeno ponto opcional de redação: o ajuste de limiar empatou com a ponderação de classes em 0,643, então "a ponderação de classes venceu" e "superando as demais técnicas na validação" não são exatos. Você poderia escrever "igualando o ajuste de limiar e superando o upsampling e o downsampling na validação".

</div>